In [1]:
import streamlit as st
import pandas as pd
import requests
import time
import json
import os
import pandas as pd
import datetime

# https://streamlit.io/cloud

In [2]:
BASE_URL = "https://codeforces.com/api/"
REQUEST_DELAY = 2
DEF_LANG = "en"
HEADERS = {
    "User-Agent": "CodeForces-API-Client/1.0",
    "Accept": "application/json"
}
ultima_req_t = 0

In [3]:
def requisicao(metodo, params={}):
    global ultima_req_t, REQUEST_DELAY, BASE_URL, HEADERS, PARAMS

    # Verificar limite de 1 requisição a cada 2s
    t_dec_ult_req = time.time() - ultima_req_t

    if t_dec_ult_req < REQUEST_DELAY:
        time.sleep(REQUEST_DELAY - t_dec_ult_req)

    url = BASE_URL + metodo

    #params["lang"] = DEF_LANG

    try:
        response = requests.get(url, params=params, headers=HEADERS)
        ultima_req_t = time.time()

        if response.status_code == 200:
            data = response.json()
            return data
        else:
            err = {"status": "FAILED", "comment": f"HTTP Error: {response.status_code}"}
            print(err)
            return err

    except Exception as e:
        err = {"status": "FAILED", "comment": f"Request failed: {str(e)}"}
        print(err)
        return err

In [4]:
def ler_handles():
    handles = []
    with open ("users.txt", encoding='utf-8') as file:
        for h in file:
            handles.append(h.strip())
    return handles

In [5]:
## Informações dos usuários
handles = ler_handles()
handles_sc = ";".join(handles)
data = requisicao("user.info", {"handles":handles_sc})
users = pd.DataFrame(data["result"])
users = users[["handle", "firstName", "rating", "rank", "lastOnlineTimeSeconds"]]
users

,handle,firstName,rating,rank,lastOnlineTimeSeconds
0,anacarlaaf,Ana Carla,760,newbie,1774383291
1,luanzito,Luan,1654,expert,1774451289
2,rebecamadi,Rebeca,1222,pupil,1774449611
3,lip33,Felipe,1261,pupil,1774388153


In [6]:
subs = pd.DataFrame()
contests = pd.DataFrame()
for h in handles:

    # Informações das submissões
    data = requisicao("user.status", {"handle": h, "count": 20})
    data = pd.DataFrame(data["result"])
    data = data.drop("author", axis=1)     # remover author (contar apenas individual, sem time)
    data["handle"] = h    # adicionar handle
    problem_df = pd.json_normalize(data["problem"]).add_prefix("problem_") # expandir problem (multivalorado)
    data = data.drop(columns=["problem"]).join(problem_df)
    subs = pd.concat([subs, data], ignore_index=True)

    # Informações dos contests
    data = requisicao("user.rating", {"handle":h})
    data = pd.DataFrame(data["result"])
    contests = pd.concat([contests, data], ignore_index=True)

KeyboardInterrupt: 

In [ ]:
subs.head()

,id,contestId,creationTimeSeconds,relativeTimeSeconds,programmingLanguage,verdict,testset,passedTestCount,timeConsumedMillis,memoryConsumedBytes,handle,problem_contestId,problem_index,problem_name,problem_type,problem_rating,problem_tags,problem_points
0,367018207,2125,1773681896,2147483647,"C++23 (GCC 14-64, msys2)",OK,TESTS,6,93,102400,anacarlaaf,2125,B,Left and Down,PROGRAMMING,900.0,"[math, number theory]",NaN
1,367017988,2125,1773681805,2147483647,"C++23 (GCC 14-64, msys2)",WRONG_ANSWER,TESTS,1,46,0,anacarlaaf,2125,B,Left and Down,PROGRAMMING,900.0,"[math, number theory]",NaN
2,366914132,1389,1773657235,2147483647,"C++23 (GCC 14-64, msys2)",WRONG_ANSWER,TESTS,0,31,0,anacarlaaf,1389,A,LCM Problem,PROGRAMMING,800.0,"[constructive algorithms, greedy, math, number...",NaN
3,366912153,2125,1773655940,2147483647,"C++23 (GCC 14-64, msys2)",WRONG_ANSWER,TESTS,1,46,0,anacarlaaf,2125,B,Left and Down,PROGRAMMING,900.0,"[math, number theory]",NaN
4,366866086,2033,1773614515,2147483647,"C++23 (GCC 14-64, msys2)",OK,TESTS,3,593,1024000,anacarlaaf,2033,B,Sakurako and Water,PROGRAMMING,900.0,"[brute force, constructive algorithms, greedy]",NaN


In [ ]:
contests.head()

,contestId,contestName,handle,rank,ratingUpdateTimeSeconds,oldRating,newRating
0,1976,Educational Codeforces Round 166 (Rated for Di...,anacarlaaf,14388,1717086900,0,391
1,1984,Codeforces Global Round 26,anacarlaaf,11007,1717954500,391,689
2,1986,Codeforces Round 954 (Div. 3),anacarlaaf,14573,1719162300,689,868
3,1996,Codeforces Round 962 (Div. 3),anacarlaaf,21702,1722013500,868,914
4,1999,Codeforces Round 964 (Div. 4),anacarlaaf,19847,1722963600,914,952


In [ ]:
df = pd.merge(users, subs, on="handle")
df = df.merge(contests, on=["handle", "contestId"], how="left")
df.head()

,handle,firstName,rating,rank_x,lastOnlineTimeSeconds,id,contestId,creationTimeSeconds,relativeTimeSeconds,programmingLanguage,...,problem_name,problem_type,problem_rating,problem_tags,problem_points,contestName,rank_y,ratingUpdateTimeSeconds,oldRating,newRating
0,anacarlaaf,Ana Carla,760,newbie,1773683010,367018207,2125,1773681896,2147483647,"C++23 (GCC 14-64, msys2)",...,Left and Down,PROGRAMMING,900.0,"[math, number theory]",NaN,Educational Codeforces Round 181 (Rated for Di...,16175.0,1.753202e+09,878.0,835.0
1,anacarlaaf,Ana Carla,760,newbie,1773683010,367017988,2125,1773681805,2147483647,"C++23 (GCC 14-64, msys2)",...,Left and Down,PROGRAMMING,900.0,"[math, number theory]",NaN,Educational Codeforces Round 181 (Rated for Di...,16175.0,1.753202e+09,878.0,835.0
2,anacarlaaf,Ana Carla,760,newbie,1773683010,366914132,1389,1773657235,2147483647,"C++23 (GCC 14-64, msys2)",...,LCM Problem,PROGRAMMING,800.0,"[constructive algorithms, greedy, math, number...",NaN,NaN,NaN,NaN,NaN,NaN
3,anacarlaaf,Ana Carla,760,newbie,1773683010,366912153,2125,1773655940,2147483647,"C++23 (GCC 14-64, msys2)",...,Left and Down,PROGRAMMING,900.0,"[math, number theory]",NaN,Educational Codeforces Round 181 (Rated for Di...,16175.0,1.753202e+09,878.0,835.0
4,anacarlaaf,Ana Carla,760,newbie,1773683010,366866086,2033,1773614515,2147483647,"C++23 (GCC 14-64, msys2)",...,Sakurako and Water,PROGRAMMING,900.0,"[brute force, constructive algorithms, greedy]",NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   handle                   80 non-null     object 
 1   firstName                80 non-null     object 
 2   rating                   80 non-null     int64  
 3   rank_x                   80 non-null     object 
 4   lastOnlineTimeSeconds    80 non-null     int64  
 5   id                       80 non-null     int64  
 6   contestId                80 non-null     int64  
 7   creationTimeSeconds      80 non-null     int64  
 8   relativeTimeSeconds      80 non-null     int64  
 9   programmingLanguage      80 non-null     object 
 10  verdict                  80 non-null     object 
 11  testset                  80 non-null     object 
 12  passedTestCount          80 non-null     int64  
 13  timeConsumedMillis       80 non-null     int64  
 14  memoryConsumedBytes      80 

In [ ]:
df_copy = df.copy()

In [12]:
df_user = requisicao("user.status", {"handle":"anacarlaaf", "count":100})
df_user = pd.DataFrame(df_user["result"])
df_user.head(30)

,id,contestId,creationTimeSeconds,relativeTimeSeconds,problem,author,programmingLanguage,verdict,testset,passedTestCount,timeConsumedMillis,memoryConsumedBytes
0,368168942,158,1774455149,2147483647,"{'contestId': 158, 'index': 'A', 'name': 'Next...","{'contestId': 158, 'participantId': 218363911,...",PyPy 3-64,OK,TESTS,50,156,0
1,368168564,158,1774454949,2147483647,"{'contestId': 158, 'index': 'A', 'name': 'Next...","{'contestId': 158, 'participantId': 218363911,...",PyPy 3-64,WRONG_ANSWER,TESTS,5,156,0
2,368125748,2178,1774433692,2147483647,"{'contestId': 2178, 'index': 'B', 'name': 'Imp...","{'contestId': 2178, 'participantId': 233732026...",PyPy 3-64,OK,TESTS,10,187,8806400
3,368120548,2178,1774430637,2147483647,"{'contestId': 2178, 'index': 'A', 'name': 'Yes...","{'contestId': 2178, 'participantId': 233732026...",PyPy 3-64,OK,TESTS,4,78,1331200
4,368120498,2178,1774430603,2147483647,"{'contestId': 2178, 'index': 'A', 'name': 'Yes...","{'contestId': 2178, 'participantId': 233732026...",PyPy 3-64,WRONG_ANSWER,TESTS,0,31,0
5,368119519,2178,1774430039,2147483647,"{'contestId': 2178, 'index': 'A', 'name': 'Yes...","{'contestId': 2178, 'participantId': 233732026...",PyPy 3-64,WRONG_ANSWER,TESTS,1,93,1536000
6,368072347,2178,1774379071,2147483647,"{'contestId': 2178, 'index': 'A', 'name': 'Yes...","{'contestId': 2178, 'participantId': 233732026...",PyPy 3-64,WRONG_ANSWER,TESTS,1,62,1536000
7,368071452,2178,1774378405,2147483647,"{'contestId': 2178, 'index': 'A', 'name': 'Yes...","{'contestId': 2178, 'participantId': 233732026...",PyPy 3-64,WRONG_ANSWER,TESTS,1,46,1331200
8,368070918,2185,1774377970,2147483647,"{'contestId': 2185, 'index': 'C', 'name': 'Shi...","{'contestId': 2185, 'participantId': 233732023...","C++23 (GCC 14-64, msys2)",OK,TESTS,11,46,0
9,368059799,2194,1774372553,2147483647,"{'contestId': 2194, 'index': 'A', 'name': 'Law...","{'contestId': 2194, 'participantId': 233710529...","C++23 (GCC 14-64, msys2)",OK,TESTS,4,31,0
